# Patchscope — open-answer (`open_summary`) analysis

Loads `patchscope_*.jsonl` from `results/raw/`, summarizes decode outputs, and checks whether
injected activations carry answer-letter information that reporters can recover.

## Key fields in the JSONL

### Source-side
| Field | Meaning |
|---|---|
| `source_last_prefill_answer` | Argmax of A/B/C/D logits at last prefill token (canonical) |
| `source_generated_answer` | Letter the model actually generated during decode (canonical) |
| `source_answer_probs` | Full `{A: p, B: p, C: p, D: p}` softmax from prefill logits (canonical) |
| `source_chosen_prob` | `max(source_answer_probs)` — confidence of prefill answer |
| `source_margin` | `top1_prob - top2_prob` — how decisive the source was |
| `source_answer_entropy` | `-sum(p * log(p))` over the 4 choices (~0 = certain, ~1.39 = uniform) |
| `source_extraction_mode` | `"prefill_last_before_assistant"` or `"during_generation"` |
| `source_extraction_token_id` | Token ID at the extraction site |
| `source_extraction_token_text` | Decoded text of that token (e.g. `"B"` for during_gen) |

### Reporter-side
| Field | Meaning |
|---|---|
| `reporter_generated_text` | Raw text the reporter produced (shuffled space, not remapped) |
| `reporter_parsed_answer` | Letter from `reporter_generated_text`, remapped to canonical |
| `reporter_is_correct` | `True` if `reporter_parsed_answer` matches the right ground truth |
| `reporter_parse_success` | Whether a letter was successfully extracted |
| `reporter_chosen_prob` | `max(reporter_choice_probs)` — reporter confidence |
| `reporter_margin` | Reporter `top1_prob - top2_prob` |
| `reporter_decode_mode` | `"logits"` or `"generate"` |
| `reporter_interpretation_prompt` | Full prompt sent to the reporter model |

### Question metadata
| Field | Meaning |
|---|---|
| `category_id` / `category_name` | Question category from task JSON |
| `expected_disagreement` | Should personas disagree here? |
| `neutral_reference_answer` | Ground truth answer from task JSON |

**Ground truth rule**: for `during_generation` records, compare `reporter_parsed_answer` against
`source_generated_answer`. For `prefill` records, compare against `source_last_prefill_answer`.
The `reporter_is_correct` field handles this automatically.


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display


USER_JSONL_INPUT = "results/raw/raw 04 02/ps_l8b_20260403_023453_whatuisutheudefiniti_pairs23-phu-Px1.jsonl"
LOGIT_LENS_INPUT = "results/raw/raw 04 02/ps_l8b_20260403_023453_LOGIT_LENS_whatuisutheudefiniti_pairs23-phu-Px1.json"

# Resolve repo root (notebook may be run from repo root or from analysis/)
ROOT = Path.cwd()
if not (ROOT / "results" / "raw").is_dir() and (ROOT.parent / "results" / "raw").is_dir():
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "results" / "raw"

def resolve_plot_model_label(path: Path, frame: pd.DataFrame) -> str:
    """HuggingFace id from records if present, else slug parsed from `patchscope_<slug>_<YYYYMMDD_HHMMSS>_*.jsonl`."""
    file_slug: str | None = None
    stem = path.stem
    if stem.startswith("patchscope_"):
        rest = stem[len("patchscope_") :]
        m = re.search(r"_(\d{8}_\d{6})_", rest)
        if m:
            file_slug = rest[: m.start()]
    record_model: str | None = None
    if len(frame) and "model" in frame.columns:
        s = frame["model"].dropna()
        if len(s):
            record_model = str(s.iloc[0])
    if record_model and file_slug:
        return f"{record_model} · file:{file_slug}"
    if record_model:
        return record_model
    if file_slug:
        return f"{file_slug} (from filename)"
    return stem


def resolve_jsonl_path(user_path: str) -> Path:
    # Try the path relative to current working dir
    p = Path(user_path)
    if p.is_file():
        return p.resolve()
    # Try relative to repo root (for notebook run from analysis/ subdir)
    p2 = (ROOT / user_path)
    if p2.is_file():
        return p2.resolve()
    # Try relative to RESULTS_DIR (results/raw/)
    p3 = (RESULTS_DIR / user_path)
    if p3.is_file():
        return p3.resolve()
    # Try finding just by filename in RESULTS_DIR/* recursively if user supplied file only
    if len(user_path.split("/")) == 1:
        found = list(RESULTS_DIR.rglob(user_path))
        if found:
            return found[0].resolve()
    return None

JSONL_PATH = resolve_jsonl_path(USER_JSONL_INPUT)

if JSONL_PATH is None or not JSONL_PATH.is_file():
    # Fallback: most recent patchscope_*.jsonl in results/raw
    candidates = sorted(RESULTS_DIR.glob("patchscope_*.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)
    JSONL_PATH = candidates[0] if candidates else None

print(f"ROOT={ROOT}")
print(f"JSONL={JSONL_PATH} (exists={JSONL_PATH is not None and JSONL_PATH.is_file()})")

# ── Logit lens ────────────────────────────────────────────────────────
LOGIT_LENS_PATH = resolve_jsonl_path(LOGIT_LENS_INPUT)  # reuse same resolver
if LOGIT_LENS_PATH is None:
    _p = Path(LOGIT_LENS_INPUT)
    if _p.is_file():
        LOGIT_LENS_PATH = _p.resolve()
print(f"LOGIT_LENS={LOGIT_LENS_PATH} (exists={LOGIT_LENS_PATH is not None and LOGIT_LENS_PATH.is_file()})")


ROOT=/Users/daylight/dev/code/cross_persona_introspection
JSONL=/Users/daylight/dev/code/cross_persona_introspection/results/raw/raw 04 02/ps_l8b_20260402_204506_repeatuword_openusummary_pairs9-phu.jsonl (exists=True)
LOGIT_LENS=/Users/daylight/dev/code/cross_persona_introspection/results/raw/raw 04 02/ps_l8b_20260403_023453_LOGIT_LENS_whatuisutheudefiniti_pairs23-phu-Px1.json (exists=True)


In [2]:
def load_patchscope_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(rows)


df = load_patchscope_jsonl(JSONL_PATH)
PLOT_MODEL_LABEL = resolve_plot_model_label(JSONL_PATH, df)
print(f"rows={len(df):,}  cols={len(df.columns)}")
print(f"Model label (plots): {PLOT_MODEL_LABEL}")
df.head(2)


rows=22,140  cols=44
Model label (plots): meta-llama/Llama-3.1-8B-Instruct


,experiment,template_name,model,question_id,source_persona,reporter_persona,condition,source_layer,injection_layer,injection_mode,...,extraction_token_id,extraction_token_text,question_text,question_options,category_id,category_name,expected_disagreement,neutral_reference_answer,error,timestamp
0,patchscope,open_summary,meta-llama/Llama-3.1-8B-Instruct,pol_001,persona_conservative,persona_conservative,real,5,5,replace,...,220,' ',A city has seen a sharp rise in open-air drug ...,{'A': 'Launch a mixed pilot with both enforcem...,cat_1,Category 1: Policy Questions (High disagreemen...,high,None,None,2026-04-02T20:45:47.222635+00:00
1,patchscope,open_summary,meta-llama/Llama-3.1-8B-Instruct,pol_001,persona_conservative,persona_progressive,real,5,5,replace,...,220,' ',A city has seen a sharp rise in open-air drug ...,{'A': 'Launch a mixed pilot with both enforcem...,cat_1,Category 1: Policy Questions (High disagreemen...,high,None,None,2026-04-02T20:45:47.333026+00:00


In [3]:
# Run health (text-first + counts)
print(f"=== Run health · {PLOT_MODEL_LABEL} ===\n")
if len(df):
    print("template_name (full):")
    print(df["template_name"].value_counts().to_string())
    print("\ncondition (full):")
    print(df["condition"].value_counts().to_string())
    print("\ndecode_mode (full):")
    print(df["reporter_decode_mode"].value_counts().to_string())
    bad_err = df["error"].fillna("").astype(str).str.strip().ne("")
    print(f"\nrows with non-empty error: {bad_err.sum()}")
    print("\n(source_layer → injection_layer) top pairs:")
    pair = df.assign(p=df["source_layer"].astype(str) + "→" + df["injection_layer"].astype(str))
    print(pair["p"].value_counts().to_string())
else:
    print("No rows loaded yet.")


=== Run health · meta-llama/Llama-3.1-8B-Instruct ===

template_name (full):
template_name
open_summary    22140

condition (full):
condition
real    22140

decode_mode (full):


KeyError: 'reporter_decode_mode'

In [ ]:
_LETTER = re.compile(r"\b([ABCD])\b", re.I)
_NONCOPY = re.compile(r"^[0-9\s\->;]+$")


def extract_predicted_letter(text: str) -> str | None:
    if not isinstance(text, str) or not text.strip():
        return None
    m = _LETTER.search(text)
    return m.group(1).upper() if m else None


def get_ground_truth(rec: pd.Series) -> str | None:
    """Return the correct ground truth based on source_extraction_mode.
    
    For during_generation: use source_generated_answer (what the model actually generated).
    For prefill: use source_last_prefill_answer (logits-based).
    """
    mode = rec.get("source_extraction_mode", "")
    if isinstance(mode, str) and "during_generation" in mode:
        ans = rec.get("source_generated_answer")
        if pd.notna(ans):
            return ans
    return rec.get("source_last_prefill_answer")


def option_text(rec: pd.Series) -> str | None:
    opts = rec.get("question_options") or {}
    key = get_ground_truth(rec)
    if not key or key not in opts:
        return None
    return opts[key]


def keyword_overlap_score(generated: str, option_blob: str | None) -> float:
    """Crude: fraction of content words from option that appear in generated (lower case)."""
    if not option_blob or not isinstance(generated, str):
        return 0.0
    stop = {"a", "the", "on", "in", "to", "and", "or", "if", "of", "for", "is", "are", "be"}
    ow = [w for w in re.findall(r"[a-zA-Z]{4,}", option_blob.lower()) if w not in stop]
    if not ow:
        return 0.0
    g = generated.lower()
    hits = sum(1 for w in ow if w in g)
    return hits / len(ow)


sub = df[df["template_name"] == "open_summary"].copy() if len(df) else df
if len(sub):
    sub["ground_truth"] = sub.apply(get_ground_truth, axis=1)
    # Use reporter_parsed_answer (canonical, remapped) as primary match
    sub["matches_ground_truth"] = sub["reporter_parsed_answer"] == sub["ground_truth"]
    # Also keep raw heuristic for comparison
    sub["predicted_letter_raw"] = sub["reporter_generated_text"].map(extract_predicted_letter)
    sub["option_snippet"] = sub.apply(option_text, axis=1)
    sub["kw_overlap"] = sub.apply(
        lambda r: keyword_overlap_score(r["reporter_generated_text"], r["option_snippet"]), axis=1
    )

    print(f"=== open_summary heuristics · {PLOT_MODEL_LABEL} ===")
    print(f"rows: {len(sub):,}")
    print(f"has reporter_parsed_answer: {sub['reporter_parsed_answer'].notna().mean():.1%}")
    print(f"extraction_modes: {sub['source_extraction_mode'].value_counts().to_dict()}")
else:
    print("No open_summary rows.")


### Heuristics (limitations)

- **`predicted_letter`**: first `A`/`B`/`C`/`D` token in `reporter_generated_text`; many completions never emit a letter (especially pattern-echo junk).
- **`kw_overlap`**: share of long words from the **source-chosen** option text that appear in the generation (rough proxy for “talking about the right stance”).
- **`pattern_echo`**: detects generations that mostly repeat `N -> N` style lines like the priming examples — often **not** a usable “summary.”


In [ ]:
if len(sub):
    print(f"=== By condition · {PLOT_MODEL_LABEL} ===")
    g = sub.groupby("condition", dropna=False).agg(
        n=("question_id", "count"),
        letter_acc=("matches_source_letter", "mean"),
        has_letter=("predicted_letter", lambda s: s.notna().mean()),
        kw_mean=("kw_overlap", "mean"),
        pattern_echo=("pattern_echo", "mean"),
    )
    g_round = g.round(4)
    print(g_round.to_string())
    print()
    display(g_round)

    print(f"\n=== Mean kw_overlap × reporter_persona · {PLOT_MODEL_LABEL} ===")
    piv = sub.pivot_table(
        index="reporter_persona",
        columns="condition",
        values="kw_overlap",
        aggfunc="mean",
    )
    piv_r = piv.round(3)
    print(piv_r.to_string())
    print()
    display(piv_r)
else:
    print("Skip aggregates — no open_summary rows.")


In [ ]:
# Paired contrast: same (question, source_persona, reporter, layers) — real minus baseline
if len(sub):
    print(f"=== Paired kw_lift (real − baseline) · {PLOT_MODEL_LABEL} ===")
    keys = ["question_id", "source_persona", "reporter_persona", "source_layer", "injection_layer"]
    wide = sub.pivot_table(
        index=keys,
        columns="condition",
        values="kw_overlap",
        aggfunc="mean",
    )
    if "real" in wide.columns and "text_only_baseline" in wide.columns:
        wide["kw_lift_real_minus_baseline"] = wide["real"] - wide["text_only_baseline"]
        lift = wide["kw_lift_real_minus_baseline"]
        print(
            "Summary: mean=%.4f  median=%.4f  std=%.4f  n_pairs=%d"
            % (lift.mean(), lift.median(), lift.std(), lift.notna().sum())
        )
        desc = lift.describe().to_frame("kw_lift")
        print("\n" + desc.to_string())
        print()
        display(desc)
    else:
        print("Need both 'real' and 'text_only_baseline' rows for paired lift.")


In [ ]:
if len(sub):
    print(f"=== kw_overlap by layer pair × condition · {PLOT_MODEL_LABEL} ===")
    pair = sub["source_layer"].astype(str) + "→" + sub["injection_layer"].astype(str)
    by_layer = (
        sub.assign(pair=pair)
        .groupby(["pair", "condition"], as_index=False)
        .agg(kw_mean=("kw_overlap", "mean"), n=("question_id", "count"))
    )
    top_pairs = sub.assign(pair=pair).groupby("pair").size().sort_values(ascending=False).head(15).index
    tbl = (
        by_layer[by_layer["pair"].isin(top_pairs)]
        .sort_values(["pair", "condition"])
        .round(4)
    )
    print("(top 15 layer pairs by row count)\n")
    print(tbl.to_string(index=False))
    print()
    display(tbl)


In [ ]:
if len(sub):
    try:
        import matplotlib.pyplot as plt

        ser = (
            sub.groupby("condition")["kw_overlap"]
            .mean()
            .reindex(["real", "text_only_baseline", "shuffled"])
        )
        print(f"=== Mean kw_overlap by condition (plotted) · {PLOT_MODEL_LABEL} ===")
        print(ser.to_string())
        print()
        fig, ax = plt.subplots(figsize=(7, 4))
        ser.plot(kind="bar", ax=ax, rot=0, color="steelblue")
        ax.set_ylabel("kw_overlap")
        ax.set_title(
            f"{PLOT_MODEL_LABEL}\nMean keyword overlap vs source-chosen option (open_summary)",
            fontsize=10,
        )
        fig.tight_layout()
        plt.show()

        piv_plot = sub.pivot_table(
            index="reporter_persona",
            columns="condition",
            values="kw_overlap",
            aggfunc="mean",
        ).round(4)
        print(f"=== Mean kw_overlap: reporter × condition (plotted) · {PLOT_MODEL_LABEL} ===")
        print(piv_plot.to_string())
        print()
        fig2, ax2 = plt.subplots(figsize=(8.5, 4))
        piv_plot.plot(kind="bar", ax=ax2, rot=0)
        ax2.set_ylabel("kw_overlap")
        ax2.set_title(
            f"{PLOT_MODEL_LABEL}\nMean kw_overlap by reporter persona × condition",
            fontsize=10,
        )
        ax2.legend(title="condition", bbox_to_anchor=(1.02, 1), loc="upper left")
        fig2.tight_layout()
        plt.show()
    except Exception as e:
        print("Plot skipped:", e)


In [ ]:
# One question, all conditions — qualitative check (first stable question_id)
if len(sub):
    print(f"=== Sample question (table + truncated text) · {PLOT_MODEL_LABEL} ===")
    qid = sorted(sub["question_id"].unique())[0]
    cols = [
        "condition",
        "source_persona",
        "reporter_persona",
        "source_layer",
        "injection_layer",
        "source_last_prefill_answer",
        "predicted_letter",
        "kw_overlap",
        "pattern_echo",
        "reporter_generated_text",
    ]
    view = sub.loc[sub["question_id"] == qid, cols].sort_values(
        ["source_layer", "injection_layer", "reporter_persona", "condition"]
    )
    print("question_id:", qid)
    qt = sub.loc[sub["question_id"] == qid, "question_text"].iloc[0]
    print("question_text:", qt[:400] + ("…" if len(qt) > 400 else ""))
    view_print = view.copy()
    view_print["reporter_generated_text"] = view_print["reporter_generated_text"].astype(str).str.slice(0, 100) + "…"
    print("\n" + view_print.to_string(index=False))
    print()
    display(view)


## Letter-match analysis

The identity prompt (`cat -> cat; 135 -> 135; {placeholder} ->`) is designed to make the model echo back whatever information is encoded in the patched activation. The key metric is whether the **answer letter** (A/B/C/D) that leaks through the pattern-echo output matches the source persona's direct answer.

- **letter_acc** = fraction where predicted letter matches `source_last_prefill_answer`
- Chance level = 25% (random among 4 choices)
- **real > chance** = the patched activation carries answer information
- **real > shuffled** = the information is specific to *this* question, not generic persona signal
- **baseline = 0** = without an activation, the prompt alone produces no letters (good — means letters come from the activation)


In [ ]:
# ── Statistical significance: real vs shuffled vs chance ──────────────
from scipy import stats

if len(sub):
    # Only look at rows where a letter was extracted (has_letter)
    has_letter = sub[sub["predicted_letter"].notna()].copy()

    for cond in ["real", "shuffled"]:
        cond_rows = has_letter[has_letter["condition"] == cond]
        n = len(cond_rows)
        k = cond_rows["matches_source_letter"].sum()
        if n == 0:
            print(f"{cond}: no rows with extracted letters")
            continue
        acc = k / n
        # Binomial test vs chance (25%)
        p_vs_chance = stats.binomtest(int(k), n, 0.25, alternative="greater").pvalue
        print(f"{cond}: {k}/{n} = {acc:.1%}  (p vs 25% chance = {p_vs_chance:.2e})")

    # Two-proportion z-test: real vs shuffled
    real_rows = has_letter[has_letter["condition"] == "real"]
    shuf_rows = has_letter[has_letter["condition"] == "shuffled"]
    n_real, k_real = len(real_rows), int(real_rows["matches_source_letter"].sum())
    n_shuf, k_shuf = len(shuf_rows), int(shuf_rows["matches_source_letter"].sum())
    if n_real > 0 and n_shuf > 0:
        p_real, p_shuf = k_real / n_real, k_shuf / n_shuf
        p_pool = (k_real + k_shuf) / (n_real + n_shuf)
        se = (p_pool * (1 - p_pool) * (1/n_real + 1/n_shuf)) ** 0.5
        z = (p_real - p_shuf) / se if se > 0 else 0
        p_two_sided = 2 * (1 - stats.norm.cdf(abs(z)))
        print(f"\nreal vs shuffled: {p_real:.1%} vs {p_shuf:.1%}")
        print(f"  z = {z:.2f}, p = {p_two_sided:.2e} (two-sided)")
        print(f"  lift = {p_real - p_shuf:+.1%}")
else:
    print("No data.")


In [ ]:
# ── Letter-match accuracy by layer pair ───────────────────────────────
# Which (source_layer → injection_layer) combinations carry the most answer signal?

if len(sub):
    import matplotlib.pyplot as plt

    real_sub = sub[sub["condition"] == "real"].copy()
    real_sub["layer_pair"] = (
        real_sub["source_layer"].astype(str) + "→" + real_sub["injection_layer"].astype(str)
    )

    layer_acc = (
        real_sub.groupby("layer_pair", as_index=False)
        .agg(
            letter_acc=("matches_source_letter", "mean"),
            has_letter=("predicted_letter", lambda s: s.notna().mean()),
            n=("question_id", "count"),
        )
        .sort_values("letter_acc", ascending=False)
        .round(4)
    )

    print(f"=== Letter-match accuracy by layer pair (real condition) · {PLOT_MODEL_LABEL} ===")
    print(layer_acc.to_string(index=False))
    print()
    display(layer_acc)

    # Also show shuffled for comparison
    shuf_sub = sub[sub["condition"] == "shuffled"].copy()
    shuf_sub["layer_pair"] = (
        shuf_sub["source_layer"].astype(str) + "→" + shuf_sub["injection_layer"].astype(str)
    )
    shuf_acc = shuf_sub.groupby("layer_pair", as_index=False).agg(
        letter_acc_shuffled=("matches_source_letter", "mean"),
    ).round(4)

    merged = layer_acc.merge(shuf_acc, on="layer_pair", how="left")
    merged["lift_over_shuffled"] = (merged["letter_acc"] - merged["letter_acc_shuffled"]).round(4)

    print(f"\n=== Real vs shuffled by layer pair · {PLOT_MODEL_LABEL} ===")
    print(merged[["layer_pair", "letter_acc", "letter_acc_shuffled", "lift_over_shuffled", "n"]].to_string(index=False))

    # Plot
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = range(len(merged))
    width = 0.35
    ax.bar([i - width/2 for i in x], merged["letter_acc"], width, label="real", color="steelblue")
    ax.bar([i + width/2 for i in x], merged["letter_acc_shuffled"], width, label="shuffled", color="salmon")
    ax.axhline(0.25, color="gray", linestyle="--", linewidth=1, label="chance (25%)")
    ax.set_xticks(list(x))
    ax.set_xticklabels(merged["layer_pair"], rotation=45, ha="right")
    ax.set_ylabel("Letter-match accuracy")
    ax.set_xlabel("source_layer → injection_layer")
    ax.set_title(f"{PLOT_MODEL_LABEL}\nAnswer letter recovery by layer pair", fontsize=10)
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No data.")


In [ ]:
# ── Letter-match by source_persona × reporter_persona ─────────────────
# Does a conservative reporter decode a conservative source's answer better
# than a progressive source's answer?  This is the core PSM question:
# if personas change computation, same-persona pairs should show higher accuracy.

if len(sub):
    import matplotlib.pyplot as plt

    real_sub = sub[(sub["condition"] == "real") & sub["predicted_letter"].notna()].copy()

    cross = (
        real_sub.groupby(["source_persona", "reporter_persona"], as_index=False)
        .agg(
            letter_acc=("matches_source_letter", "mean"),
            n=("question_id", "count"),
        )
        .round(4)
    )

    # Add same/cross persona label
    cross["pair_type"] = cross.apply(
        lambda r: "same" if r["source_persona"] == r["reporter_persona"] else "cross",
        axis=1,
    )

    print(f"=== Letter-match: source × reporter (real, has_letter) · {PLOT_MODEL_LABEL} ===")
    print(cross.to_string(index=False))
    print()

    # Aggregate same vs cross
    same_cross = cross.groupby("pair_type").agg(
        mean_acc=("letter_acc", "mean"),
        total_n=("n", "sum"),
    ).round(4)
    print("Same-persona vs cross-persona (mean across pairs):")
    print(same_cross.to_string())
    print()
    display(same_cross)

    # Heatmap
    piv = cross.pivot(index="source_persona", columns="reporter_persona", values="letter_acc")
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(piv.values, cmap="Blues", vmin=0, vmax=max(0.6, piv.values.max()))
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels(piv.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index)
    ax.set_xlabel("Reporter persona")
    ax.set_ylabel("Source persona")
    ax.set_title(
        f"{PLOT_MODEL_LABEL}\nLetter-match accuracy: source × reporter (real condition)",
        fontsize=10,
    )
    # Annotate cells
    for i in range(len(piv.index)):
        for j in range(len(piv.columns)):
            val = piv.values[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    color="white" if val > 0.4 else "black", fontsize=11)
    fig.colorbar(im, ax=ax, label="letter_acc")
    fig.tight_layout()
    plt.show()

    # Also show shuffled for comparison
    shuf_real = sub[(sub["condition"] == "shuffled") & sub["predicted_letter"].notna()].copy()
    if len(shuf_real):
        shuf_cross = (
            shuf_real.groupby(["source_persona", "reporter_persona"], as_index=False)
            .agg(letter_acc_shuffled=("matches_source_letter", "mean"))
            .round(4)
        )
        print(f"\n=== Same table for shuffled condition (control) ===")
        print(shuf_cross.to_string(index=False))
else:
    print("No data.")


## Answer & prediction distributions

The source × reporter heatmap shows the **reporter persona has no effect** — each row is flat (same accuracy regardless of reporter). But the **source persona matters**: conservative ~44%, progressive ~26%.

Before concluding this is a persona effect, we need to check for confounds:
- Does the model have a **letter bias** in its predictions (e.g. always predicting A)?
- Do the two personas have different **answer distributions** (e.g. conservative answers A more)?
- If the model's prediction bias happens to align with one persona's answer distribution, that alone could explain the accuracy gap.


In [ ]:
# ── Distribution of ALL first tokens in generated text ─────────────────
# Not just A/B/C/D — also I, It, The, numbers, etc.
# This reveals whether the model is actually doing pattern completion
# or producing conversational responses (e.g. "I" = "I think...").

if len(sub):
    import matplotlib.pyplot as plt

    def first_token(text):
        if not isinstance(text, str) or not text.strip():
            return None
        # Split on whitespace and common delimiters
        tok = text.strip().split()[0] if text.strip() else None
        # Clean trailing punctuation for grouping
        if tok:
            tok = tok.rstrip('.,;:->!?')
        return tok if tok else None

    sub_copy = sub.copy()
    sub_copy["first_token"] = sub_copy["reporter_generated_text"].apply(first_token)

    conditions = [c for c in ["real", "text_only_baseline", "shuffled"] if c in sub_copy["condition"].values]

    for cond in conditions:
        cond_rows = sub_copy[sub_copy["condition"] == cond]
        counts = cond_rows["first_token"].value_counts().head(15)
        print(f"\n=== First token distribution: {cond} (top 15) ===")
        for tok, n in counts.items():
            pct = n / len(cond_rows) * 100
            print(f"  {tok!r:>10s}  {n:4d}  ({pct:5.1f}%)")

    # Plot side by side
    fig, axes = plt.subplots(1, len(conditions), figsize=(5 * len(conditions), 5), sharey=True)
    if len(conditions) == 1:
        axes = [axes]

    # Use union of top 10 tokens across conditions for consistent x-axis
    all_top = set()
    for cond in conditions:
        cond_rows = sub_copy[sub_copy["condition"] == cond]
        all_top |= set(cond_rows["first_token"].value_counts().head(10).index)
    all_top = sorted(all_top, key=lambda t: -sub_copy["first_token"].value_counts().get(t, 0))

    for ax, cond in zip(axes, conditions):
        cond_rows = sub_copy[sub_copy["condition"] == cond]
        counts = cond_rows["first_token"].value_counts()
        vals = [counts.get(t, 0) / len(cond_rows) for t in all_top]
        colors = ["steelblue" if t in "ABCD" else "coral" for t in all_top]
        ax.bar(range(len(all_top)), vals, color=colors)
        ax.set_xticks(range(len(all_top)))
        ax.set_xticklabels(all_top, rotation=45, ha="right", fontsize=8)
        ax.set_title(f"{cond} (n={len(cond_rows)})", fontsize=10)
        if ax == axes[0]:
            ax.set_ylabel("Fraction of responses")

    fig.suptitle(f"{PLOT_MODEL_LABEL}\nFirst token in generated text (blue=ABCD, coral=other)",
                 fontsize=11, y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("No data.")


# ── Ground-truth vs “predicted letter” (A/B/C/D from generation) ─────────────
#
# `source_last_prefill_answer`: the letter the *source persona* chose on the MCQ (dataset).
#
# `predicted_letter`: heuristic on the *reporter’s* `reporter_generated_text`: first standalone
#   A/B/C/D token (`extract_predicted_letter` above). Not the model’s softmax over
#   options — just “did an
#   answer letter appear in the free-form decode?”. Rows with no such token are
#   excluded from the predicted-letter marginals below.
#
# Personas: `source_persona` = whose activation was patched; `reporter_persona` =
#   framing of the decoder that produced `reporter_generated_text`. Both can shift letter rates.
#
# Source-answer bars: one row per (question_id, source_persona) so we don’t multiply
# count the same MCQ answer across reporter × layer repeats.
if len(sub):
    import matplotlib.pyplot as plt

    labels = ["A", "B", "C", "D"]
    real = sub[sub["condition"] == "real"].copy()
    src_one = real.drop_duplicates(subset=["question_id", "source_persona"])
    src_ct = (
        src_one.groupby("source_persona")["source_last_prefill_answer"]
        .value_counts()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )
    src_frac = src_ct.div(src_ct.sum(axis=1), axis=0)

    letter_real = real[real["predicted_letter"].notna()].copy()

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

    src_frac.T.plot(kind="bar", ax=ax1, rot=0, width=0.75)
    ax1.set_xlabel("Source answer (ground truth letter)")
    ax1.set_ylabel("Fraction of questions")
    ax1.set_title(f"{PLOT_MODEL_LABEL}\nSource answer by source_persona (real)")
    ax1.legend(title="source_persona", fontsize=7)
    ax1.set_ylim(0, 1)

    def _pred_by(df: pd.DataFrame, group_col: str, ax, leg_title: str) -> None:
        if len(df) == 0:
            ax.text(0.5, 0.5, "No rows with predicted_letter", ha="center", va="center")
            ax.axis("off")
            return
        pred_ct = (
            df.groupby(group_col)["predicted_letter"]
            .value_counts()
            .unstack(fill_value=0)
            .reindex(columns=labels, fill_value=0)
        )
        pred_frac = pred_ct.div(pred_ct.sum(axis=1), axis=0)
        pred_frac.T.plot(kind="bar", ax=ax, rot=0, width=0.75)
        ax.set_xlabel("predicted_letter (first A/B/C/D in generation)")
        ax.set_ylabel(f"Fraction of real rows with a letter (n={len(df)})")
        ax.legend(title=leg_title, fontsize=7)
        ax.set_ylim(0, 1)

    _pred_by(letter_real, "reporter_persona", ax2, "reporter_persona")
    ax2.set_title(f"{PLOT_MODEL_LABEL}\nPredicted letter by reporter (real)")

    _pred_by(letter_real, "source_persona", ax3, "source_persona")
    ax3.set_title(f"{PLOT_MODEL_LABEL}\nPredicted letter by source (real)")

    fig.tight_layout()
    plt.show()
else:
    print("No data.")


In [ ]:
# ── Bias-corrected accuracy: is the match real or just distribution overlap? ──
#
# If the model always predicts A, and the source answers A 50% of the time,
# you'd get 50% "accuracy" from bias alone.  We compute the expected accuracy
# under independence (prediction and answer distributions uncorrelated) and
# compare to the observed accuracy.

if len(sub):
    print(f"=== Bias-corrected letter-match · {PLOT_MODEL_LABEL} ===\n")

    for cond in ["real", "shuffled"]:
        cond_sub = sub[(sub["condition"] == cond) & sub["predicted_letter"].notna()]
        if len(cond_sub) == 0:
            continue

        labels = ["A", "B", "C", "D"]
        # Marginal distributions (as fractions)
        src_counts = cond_sub["source_last_prefill_answer"].value_counts().reindex(labels, fill_value=0)
        pred_counts = cond_sub["predicted_letter"].value_counts().reindex(labels, fill_value=0)
        n = len(cond_sub)
        src_frac = src_counts / n
        pred_frac = pred_counts / n

        # Expected accuracy under independence: sum(P(src=X) * P(pred=X)) for X in ABCD
        expected_acc = sum(src_frac[l] * pred_frac[l] for l in labels)
        observed_acc = cond_sub["matches_source_letter"].mean()
        lift = observed_acc - expected_acc

        print(f"  {cond}:")
        print(f"    observed accuracy = {observed_acc:.3f}")
        print(f"    expected (bias-only, independent) = {expected_acc:.3f}")
        print(f"    lift above bias = {lift:+.3f}")
        print(f"    → {'REAL SIGNAL' if lift > 0.02 else 'mostly bias' if lift > 0 else 'below bias'}")
        print()

    # Per source_persona
    print(f"--- Per source_persona (real condition) ---\n")
    real_with_letter = sub[(sub["condition"] == "real") & sub["predicted_letter"].notna()]
    for sp_name, sp_group in real_with_letter.groupby("source_persona"):
        src_counts = sp_group["source_last_prefill_answer"].value_counts().reindex(labels, fill_value=0)
        pred_counts = sp_group["predicted_letter"].value_counts().reindex(labels, fill_value=0)
        n = len(sp_group)
        src_frac = src_counts / n
        pred_frac = pred_counts / n
        expected_acc = sum(src_frac[l] * pred_frac[l] for l in labels)
        observed_acc = sp_group["matches_source_letter"].mean()
        lift = observed_acc - expected_acc
        print(f"  {sp_name}: observed={observed_acc:.3f}  bias_expected={expected_acc:.3f}  lift={lift:+.3f}")
    print()

    # Confusion matrix
    print(f"=== Confusion matrix (real, has_letter) · {PLOT_MODEL_LABEL} ===\n")
    real_with_letter = sub[(sub["condition"] == "real") & sub["predicted_letter"].notna()].copy()
    confusion = pd.crosstab(
        real_with_letter["source_last_prefill_answer"],
        real_with_letter["predicted_letter"],
        margins=True,
    )
    print(confusion.to_string())
    print()
    display(confusion)

    # Normalized confusion (row-wise: for each source answer, what fraction was predicted as each letter?)
    confusion_norm = pd.crosstab(
        real_with_letter["source_last_prefill_answer"],
        real_with_letter["predicted_letter"],
        normalize="index",
    ).round(3)
    print("Row-normalized (P(predicted | source_answer)):")
    print(confusion_norm.to_string())
    print()
    display(confusion_norm)

    # Heatmap of normalized confusion
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(confusion_norm.values, cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(len(confusion_norm.columns)))
    ax.set_xticklabels(confusion_norm.columns)
    ax.set_yticks(range(len(confusion_norm.index)))
    ax.set_yticklabels(confusion_norm.index)
    ax.set_xlabel("Predicted letter")
    ax.set_ylabel("Source answer")
    ax.set_title(f"{PLOT_MODEL_LABEL}\nP(predicted | source answer), real condition", fontsize=10)
    for i in range(len(confusion_norm.index)):
        for j in range(len(confusion_norm.columns)):
            val = confusion_norm.values[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    color="white" if val > 0.4 else "black", fontsize=11)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.show()
else:
    print("No data.")


## Activation impact analysis

The core question: **does injecting the activation actually change what the model predicts?**

If real, shuffled, and baseline all predict the same letter distribution (e.g. always "A"), the activation injection is doing nothing — the identity prompt's own token bias dominates. We test this three ways:

1. **Prediction distribution by condition** — side-by-side comparison of what letters each condition produces.
2. **Per-question prediction shifts** — for each question, does the real activation change the predicted letter vs baseline?
3. **Conditional accuracy on shifted predictions** — when the activation DOES shift the prediction away from baseline, is the new prediction correct more often than chance?


In [ ]:
# ── 1. Predicted letter distribution: real vs baseline vs shuffled ──────
# If all three conditions produce the same distribution, the activation
# injection has no effect — the identity prompt's letter bias dominates.

if len(sub):
    import matplotlib.pyplot as plt
    import numpy as np

    labels = ["A", "B", "C", "D"]
    conditions = [c for c in ["real", "text_only_baseline", "shuffled"] if c in sub["condition"].values]
    letter_rows = sub[sub["predicted_letter"].notna()].copy()

    fig, axes = plt.subplots(1, len(conditions), figsize=(4.5 * len(conditions), 4), sharey=True)
    if len(conditions) == 1:
        axes = [axes]

    for ax, cond in zip(axes, conditions):
        cond_rows = letter_rows[letter_rows["condition"] == cond]
        counts = cond_rows["predicted_letter"].value_counts().reindex(labels, fill_value=0)
        frac = counts / counts.sum() if counts.sum() > 0 else counts
        ax.bar(labels, frac, color="steelblue")
        ax.axhline(0.25, color="gray", ls="--", lw=1, label="uniform")
        ax.set_title(f"{cond}\n(n={len(cond_rows)} with letter)", fontsize=10)
        ax.set_ylim(0, 1)
        ax.set_xlabel("Predicted letter")
        if ax == axes[0]:
            ax.set_ylabel("Fraction")

    fig.suptitle(f"{PLOT_MODEL_LABEL}\nPredicted letter distribution by condition", fontsize=11, y=1.02)
    fig.tight_layout()
    plt.show()

    # Chi-squared test: are the distributions different?
    from scipy.stats import chi2_contingency
    if len(conditions) >= 2:
        contingency = pd.DataFrame({
            cond: letter_rows[letter_rows["condition"] == cond]["predicted_letter"]
                  .value_counts().reindex(labels, fill_value=0)
            for cond in conditions
        })
        if contingency.sum().sum() > 0:
            chi2, p, dof, _ = chi2_contingency(contingency.values.T)
            print(f"Chi-squared test (are prediction distributions different across conditions?):")
            print(f"  chi2={chi2:.2f}, dof={dof}, p={p:.2e}")
            print(f"  → {'Distributions DIFFER' if p < 0.05 else 'No significant difference — activation may not be changing predictions'}")
else:
    print("No data.")


In [ ]:
# ── 2. Per-question prediction shifts: real vs baseline (or shuffled) ────
# For each (question, source_persona, reporter, layers) tuple, check whether
# the real activation changed the predicted letter vs the control condition.
# Falls back to shuffled if text_only_baseline is not present.

if len(sub):
    import matplotlib.pyplot as plt

    keys = ["question_id", "source_persona", "reporter_persona", "source_layer", "injection_layer"]

    # Pick the best available control condition
    if "text_only_baseline" in sub["condition"].values:
        control_cond = "text_only_baseline"
    elif "shuffled" in sub["condition"].values:
        control_cond = "shuffled"
    else:
        control_cond = None

    if control_cond is None:
        print("No baseline or shuffled condition in data — enable controls in patchscope.yaml and re-run.")
    else:
        real_preds = (
            sub[sub["condition"] == "real"][[*keys, "predicted_letter", "source_last_prefill_answer"]]
            .rename(columns={"predicted_letter": "pred_real"})
        )
        ctrl_preds = (
            sub[sub["condition"] == control_cond][[*keys, "predicted_letter"]]
            .rename(columns={"predicted_letter": "pred_ctrl"})
        )

        paired = real_preds.merge(ctrl_preds, on=keys, how="inner")
        paired = paired[paired["pred_real"].notna() & paired["pred_ctrl"].notna()]

        if len(paired) > 0:
            paired["shifted"] = paired["pred_real"] != paired["pred_ctrl"]
            paired["real_correct"] = paired["pred_real"] == paired["source_last_prefill_answer"]
            paired["ctrl_correct"] = paired["pred_ctrl"] == paired["source_last_prefill_answer"]

            n_total = len(paired)
            n_shifted = paired["shifted"].sum()
            n_stayed = n_total - n_shifted

            print(f"=== Prediction shifts: real vs {control_cond} · {PLOT_MODEL_LABEL} ===")
            print(f"  Total paired rows: {n_total}")
            print(f"  Predictions that CHANGED: {n_shifted} ({n_shifted/n_total:.1%})")
            print(f"  Predictions that STAYED:  {n_stayed} ({n_stayed/n_total:.1%})")
            print()

            shifted_df = paired[paired["shifted"]]
            if len(shifted_df) > 0:
                wrong_to_right = ((~shifted_df["ctrl_correct"]) & shifted_df["real_correct"]).sum()
                right_to_wrong = (shifted_df["ctrl_correct"] & (~shifted_df["real_correct"])).sum()
                wrong_to_wrong = ((~shifted_df["ctrl_correct"]) & (~shifted_df["real_correct"])).sum()
                right_to_right = (shifted_df["ctrl_correct"] & shifted_df["real_correct"]).sum()
                print(f"  Among shifted predictions:")
                print(f"    wrong→RIGHT (activation helped):  {wrong_to_right}")
                print(f"    right→WRONG (activation hurt):    {right_to_wrong}")
                print(f"    wrong→wrong (changed, still bad): {wrong_to_wrong}")
                print(f"    right→right (changed, still ok):  {right_to_right}")
                print()

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

            ax1.pie([n_stayed, n_shifted], labels=[f"Same as {control_cond}", "Changed"],
                    colors=["lightgray", "steelblue"], autopct="%1.0f%%", startangle=90)
            ax1.set_title(f"Did activation change the prediction?\n(n={n_total})")

            acc_real = paired["real_correct"].mean()
            acc_ctrl = paired["ctrl_correct"].mean()
            bars = ax2.bar([control_cond, "Real activation"], [acc_ctrl, acc_real],
                           color=["salmon", "steelblue"])
            ax2.axhline(0.25, color="gray", ls="--", lw=1)
            ax2.set_ylabel("Letter-match accuracy")
            ax2.set_ylim(0, max(0.6, max(acc_real, acc_ctrl) + 0.1))
            ax2.set_title(f"Accuracy: {control_cond} vs real\n(paired, n={n_total})")
            for bar, val in zip(bars, [acc_ctrl, acc_real]):
                ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.1%}",
                         ha="center", fontsize=11)

            fig.suptitle(PLOT_MODEL_LABEL, fontsize=11, y=1.02)
            fig.tight_layout()
            plt.show()
        else:
            print(f"Could not pair real and {control_cond} rows (no overlap on keys, or no letters extracted).")
            print(f"  real rows: {len(sub[sub['condition'] == 'real'])}")
            print(f"  {control_cond} rows: {len(sub[sub['condition'] == control_cond])}")
            print(f"  Unique conditions in data: {sub['condition'].unique().tolist()}")
else:
    print("No data.")


In [ ]:
# ── 3. Conditional accuracy: shifted vs unshifted predictions ──────────
# If the activation carries real information, rows where it CHANGED the
# prediction should be more accurate than rows where it didn't.

if len(sub) and 'paired' in dir() and len(paired) > 0 and 'shifted' in paired.columns:
    print(f"=== Conditional accuracy · {PLOT_MODEL_LABEL} ===")
    print()

    for label, mask in [("shifted (activation changed prediction)", paired["shifted"]),
                         ("unshifted (same as control)", ~paired["shifted"])]:
        subset = paired[mask]
        if len(subset) == 0:
            print(f"  {label}: no rows")
            continue
        acc = subset["real_correct"].mean()
        print(f"  {label}:")
        print(f"    n={len(subset)}, accuracy={acc:.1%} (chance=25%)")

    print()
    shifted_acc = paired.loc[paired["shifted"], "real_correct"].mean() if paired["shifted"].any() else 0
    unshifted_acc = paired.loc[~paired["shifted"], "real_correct"].mean() if (~paired["shifted"]).any() else 0
    if shifted_acc > unshifted_acc + 0.05:
        print("  → Shifted predictions are MORE accurate: activation is carrying useful signal.")
    elif shifted_acc < unshifted_acc - 0.05:
        print("  → Shifted predictions are LESS accurate: activation may be injecting noise.")
    else:
        print("  → Similar accuracy whether shifted or not: inconclusive.")

    if paired["shifted"].any():
        shifted_df = paired[paired["shifted"]]
        print(f"\n=== Transition matrix (control→real) for shifted rows ===")
        trans = pd.crosstab(
            shifted_df["pred_ctrl"].rename("control"),
            shifted_df["pred_real"].rename("real"),
            margins=True
        )
        print(trans.to_string())
        display(trans)
else:
    print("No paired data available — enable text_only_baseline or shuffled_activation in patchscope.yaml and re-run.")


### Interpretation guide

**Letter-match accuracy** is the primary metric for the identity prompt format. The pattern-echo output (`N -> N; ...`) is expected — it's the prompt design. What matters is whether answer letters leak through the echo and match the source's answer.

Key comparisons:
- **real vs chance (25%)**: Does the patched activation carry answer information at all?
- **real vs shuffled**: Is the information specific to this question, or generic?
- **same-persona vs cross-persona pairs**: Does the reporter decode better when it shares the source's persona? (PSM signal)
- **Layer pair variation**: Which extraction → injection layer combinations carry the most signal? Mid-to-late extraction layers typically work best.

### Next steps

- Try **wrong-persona, same-question** control (patches activation from a different persona answering the same question) to isolate persona-specific signal from content signal.
- Add **random noise** control (Gaussian, norm-matched) as a sanity check.
- For free-form experiments: switch to a descriptive template and use LLM-judge or embedding similarity for evaluation.


## Extraction mode comparison

Compare `prefill` vs `during_generation` extraction sites. The activation
captured during generation should carry stronger answer signal since it's
taken at the actual answer token, not at a prompt-boundary token.


In [ ]:
# ── Accuracy split by source_extraction_mode ────────────────────────────────
if len(sub) and 'source_extraction_mode' in sub.columns:
    import matplotlib.pyplot as plt

    real = sub[sub['condition'] == 'real'].copy()
    if real['source_extraction_mode'].nunique() > 1:
        by_mode = real.groupby('source_extraction_mode').agg(
            n=('question_id', 'count'),
            match_rate=('matches_ground_truth', 'mean'),
            has_answer=('reporter_parsed_answer', lambda s: s.notna().mean()),
        ).round(4)
        print(f'=== Accuracy by extraction mode (real) · {PLOT_MODEL_LABEL} ===')
        print(by_mode.to_string())
        display(by_mode)

        fig, ax = plt.subplots(figsize=(7, 4))
        by_mode['match_rate'].plot(kind='bar', ax=ax, color='steelblue', rot=0)
        ax.axhline(0.25, color='gray', ls='--', label='chance (25%)')
        ax.set_ylabel('Match rate vs ground truth')
        ax.set_title(f'{PLOT_MODEL_LABEL}\nAccuracy by extraction mode (real)')
        ax.legend()
        fig.tight_layout()
        plt.show()
    else:
        mode = real['source_extraction_mode'].iloc[0] if len(real) else 'N/A'
        acc = real['matches_ground_truth'].mean() if len(real) else 0
        print(f'Single extraction mode: {mode}, match_rate={acc:.4f}')
else:
    print('No data or no source_extraction_mode column.')


## Source confidence & entropy analysis

Does the reporter recover the answer more accurately when the source model
was **confident** (low entropy, high margin)? If so, the activation carries
stronger signal when the model "knows" the answer.


In [ ]:
# ── Reporter accuracy vs source confidence ──────────────────────────
if len(sub) and 'source_answer_entropy' in sub.columns:
    import matplotlib.pyplot as plt
    import numpy as np

    real = sub[(sub['condition'] == 'real') & sub['reporter_parsed_answer'].notna()].copy()
    if len(real) and real['source_answer_entropy'].notna().any():
        # Bin by entropy
        real['entropy_bin'] = pd.qcut(real['source_answer_entropy'], q=4, duplicates='drop')
        by_entropy = real.groupby('entropy_bin', observed=True).agg(
            n=('question_id', 'count'),
            match_rate=('matches_ground_truth', 'mean'),
            mean_entropy=('source_answer_entropy', 'mean'),
            mean_margin=('source_margin', 'mean'),
        ).round(4)
        print(f'=== Accuracy by source entropy quartile (real) · {PLOT_MODEL_LABEL} ===')
        print(by_entropy.to_string())
        display(by_entropy)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

        # Entropy scatter
        ax1.scatter(real['source_answer_entropy'], real['matches_ground_truth'].astype(float),
                   alpha=0.15, s=20)
        # Rolling average
        sorted_r = real.sort_values('source_answer_entropy')
        window = max(20, len(sorted_r) // 10)
        rolling_acc = sorted_r['matches_ground_truth'].rolling(window, center=True).mean()
        ax1.plot(sorted_r['source_answer_entropy'], rolling_acc, 'r-', lw=2, label=f'rolling avg (w={window})')
        ax1.axhline(0.25, color='gray', ls='--', alpha=0.5, label='chance')
        ax1.set_xlabel('Source answer entropy')
        ax1.set_ylabel('Match rate')
        ax1.set_title('Reporter accuracy vs source entropy')
        ax1.legend(fontsize=8)

        # Margin scatter
        if real['source_margin'].notna().any():
            ax2.scatter(real['source_margin'], real['matches_ground_truth'].astype(float),
                       alpha=0.15, s=20)
            sorted_m = real.sort_values('source_margin')
            rolling_m = sorted_m['matches_ground_truth'].rolling(window, center=True).mean()
            ax2.plot(sorted_m['source_margin'], rolling_m, 'r-', lw=2, label=f'rolling avg')
            ax2.axhline(0.25, color='gray', ls='--', alpha=0.5)
            ax2.set_xlabel('Source margin (top1 - top2 prob)')
            ax2.set_ylabel('Match rate')
            ax2.set_title('Reporter accuracy vs source margin')
            ax2.legend(fontsize=8)

        fig.suptitle(PLOT_MODEL_LABEL, fontsize=11)
        fig.tight_layout()
        plt.show()
    else:
        print('No entropy data in real condition rows.')
else:
    print('No data or no source_answer_entropy column.')


## Category & expected disagreement breakdown

Do reporters recover answers better for certain question categories?
And does the expected level of persona disagreement affect extraction accuracy?


In [ ]:
# ── Accuracy by category and expected_disagreement ──────────────────
if len(sub):
    import matplotlib.pyplot as plt

    real = sub[(sub['condition'] == 'real') & sub['reporter_parsed_answer'].notna()].copy()

    # By category
    if 'category_id' in real.columns and real['category_id'].notna().any():
        by_cat = real.groupby(['category_id', 'category_name'], dropna=False).agg(
            n=('question_id', 'count'),
            match_rate=('matches_ground_truth', 'mean'),
            mean_entropy=('source_answer_entropy', 'mean'),
        ).round(4)
        print(f'=== Accuracy by category (real) · {PLOT_MODEL_LABEL} ===')
        print(by_cat.to_string())
        display(by_cat)

    # By expected_disagreement
    if 'expected_disagreement' in real.columns and real['expected_disagreement'].notna().any():
        by_disagree = real.groupby('expected_disagreement').agg(
            n=('question_id', 'count'),
            match_rate=('matches_ground_truth', 'mean'),
            mean_entropy=('source_answer_entropy', 'mean'),
        ).round(4)
        print(f'\n=== Accuracy by expected_disagreement (real) · {PLOT_MODEL_LABEL} ===')
        print(by_disagree.to_string())
        display(by_disagree)

        fig, ax = plt.subplots(figsize=(7, 4))
        by_disagree['match_rate'].plot(kind='bar', ax=ax, color='steelblue', rot=0)
        ax.axhline(0.25, color='gray', ls='--', label='chance')
        ax.set_ylabel('Match rate')
        ax.set_title(f'{PLOT_MODEL_LABEL}\nAccuracy by expected disagreement')
        ax.legend()
        fig.tight_layout()
        plt.show()
else:
    print('No data.')


## Prefill logits vs actual generation agreement

How often do `source_last_prefill_answer` and `source_generated_answer` agree?
When they disagree, the model's "inclination" at the prompt boundary differs from
what it actually produces — important for interpreting extraction results.


In [ ]:
# ── Prefill vs generation answer agreement ──────────────────────────
if len(sub) and 'source_generated_answer' in sub.columns:
    real = sub[sub['condition'] == 'real'].copy()
    both = real[real['source_generated_answer'].notna() & real['source_last_prefill_answer'].notna()]
    # Deduplicate to one row per (question, source_persona)
    dedup = both.drop_duplicates(subset=['question_id', 'source_persona'])
    if len(dedup):
        agree = (dedup['source_last_prefill_answer'] == dedup['source_generated_answer']).mean()
        print(f'=== Prefill vs generation agreement · {PLOT_MODEL_LABEL} ===')
        print(f'Agreement rate: {agree:.1%} ({int(agree * len(dedup))}/{len(dedup)})')
        print()

        # Show disagreements
        disagree = dedup[dedup['source_last_prefill_answer'] != dedup['source_generated_answer']]
        if len(disagree):
            print(f'Disagreements ({len(disagree)} cases):')
            cols = ['question_id', 'source_persona', 'source_last_prefill_answer',
                    'source_generated_answer', 'source_answer_entropy', 'source_margin']
            cols = [c for c in cols if c in disagree.columns]
            display(disagree[cols].head(20))

            # Are disagreements associated with higher entropy?
            if 'source_answer_entropy' in dedup.columns:
                agree_ent = dedup[dedup['source_last_prefill_answer'] == dedup['source_generated_answer']]['source_answer_entropy'].mean()
                disagree_ent = disagree['source_answer_entropy'].mean()
                print(f'\nMean entropy — agree: {agree_ent:.3f}, disagree: {disagree_ent:.3f}')
        else:
            print('Perfect agreement between prefill logits and generation.')
    else:
        print('No rows with both prefill and generated answers.')
else:
    print('No source_generated_answer column.')


## Reporter accuracy vs neutral reference answer

Does the reporter recover the **persona's answer** (which may differ from the
"neutral" correct answer), or does it just recover the objectively correct answer?
If reporters match the persona answer even when it deviates from neutral, that's
evidence the activation carries persona-specific computation.


In [ ]:
# ── Reporter vs persona answer vs neutral reference ──────────────────
if len(sub) and 'neutral_reference_answer' in sub.columns:
    real = sub[(sub['condition'] == 'real') & sub['reporter_parsed_answer'].notna()].copy()
    has_ref = real[real['neutral_reference_answer'].notna()]

    if len(has_ref):
        # Does the source persona agree with the neutral reference?
        has_ref = has_ref.copy()
        has_ref['source_agrees_neutral'] = has_ref['ground_truth'] == has_ref['neutral_reference_answer']
        has_ref['reporter_matches_persona'] = has_ref['matches_ground_truth']
        has_ref['reporter_matches_neutral'] = has_ref['reporter_parsed_answer'] == has_ref['neutral_reference_answer']

        print(f'=== Reporter recovery: persona answer vs neutral reference · {PLOT_MODEL_LABEL} ===')
        print(f'Source agrees with neutral: {has_ref["source_agrees_neutral"].mean():.1%}')
        print(f'Reporter matches persona answer: {has_ref["reporter_matches_persona"].mean():.1%}')
        print(f'Reporter matches neutral answer: {has_ref["reporter_matches_neutral"].mean():.1%}')
        print()

        # Split: when persona deviates from neutral, which does reporter follow?
        deviate = has_ref[~has_ref['source_agrees_neutral']]
        if len(deviate):
            print(f'When persona DEVIATES from neutral ({len(deviate)} rows):')
            print(f'  Reporter matches persona: {deviate["reporter_matches_persona"].mean():.1%}')
            print(f'  Reporter matches neutral: {deviate["reporter_matches_neutral"].mean():.1%}')
            print(f'  → Reporter follows {"PERSONA" if deviate["reporter_matches_persona"].mean() > deviate["reporter_matches_neutral"].mean() else "NEUTRAL"} more often')
        else:
            print('Source always agrees with neutral — no deviation cases to test.')
    else:
        print('No rows with neutral_reference_answer.')
else:
    print('No data or no neutral_reference_answer column.')


## Logit Lens Analysis

Project intermediate hidden states through the unembedding matrix at every layer.
Shows what each layer "thinks" the next token is — top-N most likely and bottom-N least likely.
Compares across personas to see whether persona conditioning changes internal representations
or only affects final-layer output.

In [ ]:
# ── Load logit lens results ──────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

ll_data = None
if LOGIT_LENS_PATH and LOGIT_LENS_PATH.is_file():
    with open(LOGIT_LENS_PATH) as f:
        ll_data = json.load(f)
    ll_config = ll_data["config"]
    ll_records = ll_data["records"]
    print(f"Logit lens: {len(ll_records)} records, {len(ll_config['layers'])} layers")
    print(f"  top_n={ll_config['top_n']}, bottom_n={ll_config['bottom_n']}")
    print(f"  phases={ll_config['phases']}")
    
    # Build a flat DataFrame: one row per (record, layer)
    ll_rows = []
    for rec in ll_records:
        for layer_entry in rec["layers"]:
            row = {
                "phase": rec["phase"],
                "persona": rec["persona"],
                "question_id": rec["question_id"],
                "question_text": rec["question_text"][:80],
                "layer": layer_entry["layer"],
                "token_text": layer_entry["token_text"],
                "token_position": layer_entry["token_position"],
                "top1_token": layer_entry["top_tokens"][0]["token"] if layer_entry["top_tokens"] else None,
                "top1_prob": layer_entry["top_tokens"][0]["prob"] if layer_entry["top_tokens"] else None,
                "top1_logit": layer_entry["top_tokens"][0]["logit"] if layer_entry["top_tokens"] else None,
                "top5_tokens": [t["token"] for t in layer_entry["top_tokens"][:5]],
                "top5_probs": [t["prob"] for t in layer_entry["top_tokens"][:5]],
                "bottom1_token": layer_entry["bottom_tokens"][0]["token"] if layer_entry["bottom_tokens"] else None,
                "bottom1_logit": layer_entry["bottom_tokens"][0]["logit"] if layer_entry["bottom_tokens"] else None,
                "top_tokens_raw": layer_entry["top_tokens"],
                "bottom_tokens_raw": layer_entry["bottom_tokens"],
            }
            ll_rows.append(row)
    ll_df = pd.DataFrame(ll_rows)
    print(f"\nFlat DataFrame: {len(ll_df)} rows (records x layers)")
    print(f"Phases: {ll_df['phase'].unique().tolist()}")
    print(f"Personas: {ll_df['persona'].unique().tolist()}")
    print(f"Questions: {ll_df['question_id'].nunique()}")
    display(ll_df.head(3))
else:
    print("No logit lens file found — skipping logit lens analysis")

In [ ]:
# ── Top-1 token by layer: does the prediction converge? ─────────────
if ll_data and len(ll_df):
    for phase in ll_df["phase"].unique():
        phase_df = ll_df[ll_df["phase"] == phase]
        print(f"\n{'='*60}")
        print(f"Phase: {phase}")
        print(f"{'='*60}")
        
        # For each persona x question, show the top-1 token trajectory across layers
        for (persona, qid), grp in phase_df.groupby(["persona", "question_id"]):
            grp = grp.sort_values("layer")
            trajectory = grp[["layer", "top1_token", "top1_prob"]].reset_index(drop=True)
            print(f"\n  {persona} | {qid}")
            # Compact: show layer -> top1_token (prob)
            parts = []
            for _, r in trajectory.iterrows():
                tok = repr(r['top1_token']).strip("'")
                parts.append(f"L{int(r['layer']):02d}:{tok}({r['top1_prob']:.4f})")
            # Print in rows of 8
            for i in range(0, len(parts), 8):
                print(f"    {' | '.join(parts[i:i+8])}")

In [ ]:
# ── Persona divergence: where do personas start predicting differently? ──
if ll_data and len(ll_df):
    source_df = ll_df[ll_df["phase"] == "source"] if "source" in ll_df["phase"].values else ll_df
    personas = sorted(source_df["persona"].unique())
    
    if len(personas) >= 2:
        print(f"Comparing top-1 predictions: {personas[0]} vs {personas[1]}")
        print(f"{'='*70}\n")
        
        # For each question, compare top-1 token at each layer between personas
        divergence_layers = []  # layer where first disagreement occurs
        agreement_by_layer = {}
        
        for qid in source_df["question_id"].unique():
            p0 = source_df[(source_df["persona"] == personas[0]) & (source_df["question_id"] == qid)]
            p1 = source_df[(source_df["persona"] == personas[1]) & (source_df["question_id"] == qid)]
            if p0.empty or p1.empty:
                continue
            p0 = p0.sort_values("layer").set_index("layer")
            p1 = p1.sort_values("layer").set_index("layer")
            common_layers = sorted(set(p0.index) & set(p1.index))
            
            first_diverge = None
            for layer in common_layers:
                agree = p0.loc[layer, "top1_token"] == p1.loc[layer, "top1_token"]
                agreement_by_layer.setdefault(layer, []).append(agree)
                if not agree and first_diverge is None:
                    first_diverge = layer
            if first_diverge is not None:
                divergence_layers.append(first_diverge)
        
        # Plot: agreement rate by layer
        layers_sorted = sorted(agreement_by_layer.keys())
        agree_rates = [np.mean(agreement_by_layer[l]) for l in layers_sorted]
        
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.bar(layers_sorted, agree_rates, color="steelblue", alpha=0.8)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Top-1 agreement rate")
        ax.set_title(f"Logit lens: top-1 token agreement between {personas[0]} & {personas[1]}")
        ax.set_ylim(0, 1.05)
        ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        if divergence_layers:
            print(f"\nFirst divergence layer — median: {np.median(divergence_layers):.0f}, "
                  f"mean: {np.mean(divergence_layers):.1f}, "
                  f"range: [{min(divergence_layers)}, {max(divergence_layers)}]")
        else:
            print("\nNo divergence found — personas agree at top-1 across all layers")
    else:
        print(f"Only one persona ({personas}) — skipping divergence analysis")

In [ ]:
# ── Top-1 probability across layers (confidence curve) ──────────────
if ll_data and len(ll_df):
    fig, ax = plt.subplots(figsize=(12, 5))
    
    for phase in ll_df["phase"].unique():
        phase_df = ll_df[ll_df["phase"] == phase]
        for persona in sorted(phase_df["persona"].unique()):
            sub = phase_df[phase_df["persona"] == persona]
            by_layer = sub.groupby("layer")["top1_prob"].agg(["mean", "std"]).reset_index()
            label = f"{phase}/{persona}"
            ax.plot(by_layer["layer"], by_layer["mean"], marker=".", label=label, alpha=0.8)
            ax.fill_between(
                by_layer["layer"],
                by_layer["mean"] - by_layer["std"],
                by_layer["mean"] + by_layer["std"],
                alpha=0.15,
            )
    
    ax.set_xlabel("Layer")
    ax.set_ylabel("Top-1 probability (mean +/- std)")
    ax.set_title("Logit lens: model confidence by layer")
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Top-5 overlap between personas by layer ─────────────────────────
# Jaccard similarity of top-5 token sets at each layer
if ll_data and len(ll_df):
    source_df = ll_df[ll_df["phase"] == "source"] if "source" in ll_df["phase"].values else ll_df
    personas = sorted(source_df["persona"].unique())
    
    if len(personas) >= 2:
        jaccard_by_layer = {}
        for qid in source_df["question_id"].unique():
            p0 = source_df[(source_df["persona"] == personas[0]) & (source_df["question_id"] == qid)]
            p1 = source_df[(source_df["persona"] == personas[1]) & (source_df["question_id"] == qid)]
            if p0.empty or p1.empty:
                continue
            p0 = p0.sort_values("layer").set_index("layer")
            p1 = p1.sort_values("layer").set_index("layer")
            for layer in sorted(set(p0.index) & set(p1.index)):
                s0 = set(p0.loc[layer, "top5_tokens"])
                s1 = set(p1.loc[layer, "top5_tokens"])
                jaccard = len(s0 & s1) / len(s0 | s1) if (s0 | s1) else 1.0
                jaccard_by_layer.setdefault(layer, []).append(jaccard)
        
        layers_sorted = sorted(jaccard_by_layer.keys())
        mean_jaccard = [np.mean(jaccard_by_layer[l]) for l in layers_sorted]
        
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(layers_sorted, mean_jaccard, marker="o", color="darkorange", markersize=5)
        ax.fill_between(layers_sorted, mean_jaccard, alpha=0.2, color="darkorange")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Mean Jaccard similarity (top-5)")
        ax.set_title(f"Logit lens: top-5 token overlap — {personas[0]} vs {personas[1]}")
        ax.set_ylim(0, 1.05)
        plt.tight_layout()
        plt.show()
    else:
        print("Need 2+ personas for overlap analysis")

In [ ]:
# ── Sample question deep dive: full top-20 at selected layers ───────
if ll_data and len(ll_df):
    source_df = ll_df[ll_df["phase"] == "source"] if "source" in ll_df["phase"].values else ll_df
    sample_qid = source_df["question_id"].iloc[0]
    sample_layers = [0, 8, 16, 24, 31]  # early, mid-early, mid, mid-late, final
    # Clamp to available layers
    all_layers = sorted(source_df["layer"].unique())
    sample_layers = [l for l in sample_layers if l in all_layers]
    if not sample_layers:
        sample_layers = all_layers[::max(1, len(all_layers)//5)][:5]
    
    print(f"Question: {sample_qid}")
    q_df = source_df[source_df["question_id"] == sample_qid]
    if len(q_df):
        print(f"Text: {q_df['question_text'].iloc[0]}")
    print()
    
    for persona in sorted(q_df["persona"].unique()):
        print(f"\n{'─'*60}")
        print(f"Persona: {persona}")
        p_df = q_df[q_df["persona"] == persona].set_index("layer")
        for layer in sample_layers:
            if layer not in p_df.index:
                continue
            row = p_df.loc[layer]
            top_toks = row["top_tokens_raw"]
            bot_toks = row["bottom_tokens_raw"]
            print(f"\n  Layer {layer}:")
            print(f"    TOP: {', '.join(f\"{t['token']}\"({t['prob']:.4f})\" for t in top_toks[:10])}")
            print(f"    BOT: {', '.join(f\"{t['token']}\"({t['prob']:.6f})\" for t in bot_toks[:5])}")